# Mel + Chroma + Tempo

## Tanítás

In [2]:
import os
import h5py
import numpy as np
import tensorflow as tf
from tensorflow.keras import mixed_precision
from tensorflow.keras.layers import (
    Input, Conv2D, Conv2DTranspose, MaxPooling2D, BatchNormalization,
    Activation, GlobalAveragePooling2D, GlobalMaxPooling2D,
    Concatenate, Dense, Dropout, Reshape, UpSampling2D
)
from tensorflow.keras.models import Model
from tensorflow.keras.utils import Sequence
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

# --- 1. MIXED PRECISION (T4 GPU-n ~1.5-2× gyorsítás, feleannyi VRAM) ---
mixed_precision.set_global_policy('mixed_float16')

# --- ÚTVONALAK ---
H5_PATH           = "/kaggle/input/datasets/bresgbor/spotify-dataset-compressed/spotify_dataset_compressed.h5"
LOAD_PATH_AE      = "/kaggle/input/models/bresgbor/spotify-autoencoder/keras/default/1/spotify_autoencoder.keras"
SAVE_PATH_AE      = "/kaggle/working/spotify_autoencoder.keras"
SAVE_PATH_ENCODER = "/kaggle/working/spotify_encoder_only.keras"


# ==========================================
# 2. EGYÉNI RÉTEGEK
# ==========================================
class L2NormLayer(tf.keras.layers.Layer):
    """L2 normalizálás a bottleneck vektoron – koszinusz-hasonlósághoz."""
    def call(self, x):
        return tf.math.l2_normalize(x, axis=1)


# ==========================================
# 3. DATA GENERATOR (Kaggle Keras 3 Kompatibilis)
# ==========================================
class SpotifyDataGenerator(Sequence):
    def __init__(self, h5_path, split_name='train', batch_size=32, shuffle=True, **kwargs):
        # Keras 3 CPU gyorsító
        super().__init__(workers=4, use_multiprocessing=False, max_queue_size=10, **kwargs)
        
        self.h5_path    = h5_path
        self.batch_size = batch_size
        self.shuffle    = shuffle

        self.hf = h5py.File(self.h5_path, 'r')
        splits     = self.hf['ml/split'][:]
        splits_str = np.array([s.decode('utf-8') if isinstance(s, bytes) else s for s in splits])
        self.indices = np.where(splits_str == split_name)[0]

        print(f"✅ Generátor '{split_name}': {len(self.indices)} dal (Batch size: {batch_size}).")
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.indices) / self.batch_size))

    def __getitem__(self, index):
        batch_split_indices  = np.arange(
            index * self.batch_size,
            min((index + 1) * self.batch_size, len(self.indices))
        )
        batch_global_indices = np.sort(self.indices[batch_split_indices])

        X_mel    = self.hf['spectrograms/mel'][batch_global_indices]
        X_chroma = self.hf['spectrograms/chroma'][batch_global_indices]
        X_tempo  = self.hf['spectrograms/tempogram'][batch_global_indices]

        X_mel    = np.expand_dims(X_mel,    axis=-1).astype(np.float32)
        X_chroma = np.expand_dims(X_chroma, axis=-1).astype(np.float32)
        X_tempo  = np.expand_dims(X_tempo,  axis=-1).astype(np.float32)

        inputs  = {'mel_input': X_mel, 'chroma_input': X_chroma, 'tempo_input': X_tempo}
        targets = {'mel_output': X_mel, 'chroma_output': X_chroma, 'tempo_output': X_tempo}
        return inputs, targets

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices)

    def __del__(self):
        if hasattr(self, 'hf') and self.hf.id.valid:
            self.hf.close()


# ==========================================
# 4. MODELL ÉPÍTŐKOCKÁK
# ==========================================
def create_encoder_branch(x, filters, pool_sizes, branch_name):
    for i, f in enumerate(filters):
        x = Conv2D(f, kernel_size=(3, 3), padding='same', name=f'{branch_name}_enc_conv_{i+1}')(x)
        x = BatchNormalization(name=f'{branch_name}_enc_bn_{i+1}')(x)
        x = Activation('relu', name=f'{branch_name}_enc_relu_{i+1}')(x)
        x = MaxPooling2D(pool_size=pool_sizes[i], name=f'{branch_name}_enc_pool_{i+1}')(x)
    avg_pool = GlobalAveragePooling2D(name=f'{branch_name}_global_avg')(x)
    max_pool = GlobalMaxPooling2D(name=f'{branch_name}_global_max')(x)
    return Concatenate(name=f'{branch_name}_pool_concat')([avg_pool, max_pool])


def create_decoder_branch(latent_input, start_shape, upsample_sizes, filters, branch_name):
    x = Dense(int(np.prod(start_shape)), activation='relu', name=f'{branch_name}_dec_dense')(latent_input)
    x = Reshape(start_shape, name=f'{branch_name}_dec_reshape')(x)

    for i, up_size in enumerate(upsample_sizes):
        x = UpSampling2D(size=up_size, name=f'{branch_name}_dec_up_{i+1}')(x)
        x = Conv2DTranspose(filters[i], kernel_size=(3, 3), padding='same', name=f'{branch_name}_dec_deconv_{i+1}')(x)
        x = BatchNormalization(name=f'{branch_name}_dec_bn_{i+1}')(x)
        x = Activation('relu', name=f'{branch_name}_dec_relu_{i+1}')(x)

    x = Conv2D(1, kernel_size=(3, 3), padding='same', activation='linear', dtype='float32', name=f'{branch_name}_output')(x)
    return x


# ==========================================
# 5. MODELL ÖSSZEÁLLÍTÁSA (1 GPU VÁLTOZAT)
# ==========================================
print("\n🚀 Autoencoder építése (1 GPU)...")

input_mel    = Input(shape=(128, 1280, 1),  name='mel_input')
input_chroma = Input(shape=(12,  1280, 1),  name='chroma_input')
input_tempo  = Input(shape=(384, 1280, 1),  name='tempo_input')

# Encoder
branch_mel    = create_encoder_branch(input_mel, [32, 64, 128, 256], [(2, 4), (2, 4), (2, 4), (2, 2)], "mel")
branch_chroma = create_encoder_branch(input_chroma, [16, 32, 64], [(1, 4), (1, 4), (1, 4)], "chroma")
# FIGYELEM: A Tempo Encoderen plusz Poolingot kapott a memóriahiba elkerülése végett (2,4)
branch_tempo  = create_encoder_branch(input_tempo, [32, 64, 128, 256], [(2, 4), (2, 4), (2, 4), (2, 2)], "tempo")

merged = Concatenate(name='concat_all_branches')([branch_mel, branch_chroma, branch_tempo])
z = Dense(512, activation='relu', name='dense_1')(merged)
z = BatchNormalization(name='bn_dense_1')(z)
z = Dropout(0.4, name='dropout_1')(z)

bottleneck_raw = Dense(256, activation='relu', dtype='float32', name='bottleneck')(z)
bottleneck     = L2NormLayer(name='l2_norm')(bottleneck_raw)

# Decoder (Ezek most a módosított formára igazodnak)
dec_mel    = create_decoder_branch(bottleneck, start_shape=(8, 10, 128), upsample_sizes=[(2, 2), (2, 4), (2, 4), (2, 4)], filters=[128, 64, 32, 16], branch_name="mel")
dec_chroma = create_decoder_branch(bottleneck, start_shape=(12, 20, 64), upsample_sizes=[(1, 4), (1, 4), (1, 4)], filters=[32, 16, 8], branch_name="chroma")
dec_tempo  = create_decoder_branch(bottleneck, start_shape=(24, 10, 128), upsample_sizes=[(2, 2), (2, 4), (2, 4), (2, 4)], filters=[128, 64, 32, 16], branch_name="tempo")

autoencoder = Model(
    inputs=[input_mel, input_chroma, input_tempo],
    outputs={'mel_output': dec_mel, 'chroma_output': dec_chroma, 'tempo_output': dec_tempo},
    name='spotify_autoencoder'
)

autoencoder.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss={'mel_output': 'mse', 'chroma_output': 'mse', 'tempo_output': 'mse'},
    loss_weights={'mel_output': 1.0, 'chroma_output': 0.3, 'tempo_output': 1.5},
    metrics={'mel_output': 'mae', 'chroma_output': 'mae', 'tempo_output': 'mae'}
)

encoder = Model(
    inputs=[input_mel, input_chroma, input_tempo],
    outputs=bottleneck,
    name='encoder_only'
)

if os.path.exists(LOAD_PATH_AE):
    print(f"\n💾 Modell betöltése az inputból: {LOAD_PATH_AE}")
    autoencoder.load_weights(LOAD_PATH_AE)
    print("✅ Súlyok betöltve!")

autoencoder.summary()


# ==========================================
# 6. TANÍTÁS (P100-hoz hangolva)
# ==========================================
print("\n⚙️ Generátorok inicializálása...")

# A P100-on a 64-es batch size-nak kényelmesen el kell férnie
BATCH_SIZE = 64 

train_gen = SpotifyDataGenerator(H5_PATH, split_name='train', batch_size=BATCH_SIZE)
val_gen   = SpotifyDataGenerator(H5_PATH, split_name='val',   batch_size=BATCH_SIZE, shuffle=False)

callbacks = [
    ModelCheckpoint(SAVE_PATH_AE, save_best_only=True, monitor='val_loss', mode='min', verbose=1),
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1)
]

print(f"\n🔥 P100 GPU TANÍTÁS INDÍTÁSA... (Batch size: {BATCH_SIZE})")
history = autoencoder.fit(
    train_gen,
    validation_data=val_gen,
    epochs=12,
    initial_epoch=8,
    callbacks=callbacks
)

# ==========================================
# 7. ENCODER MENTÉSE
# ==========================================
encoder.save(SAVE_PATH_ENCODER)
print(f"\n✅ Encoder elmentve: {SAVE_PATH_ENCODER}")

2026-03-17 08:28:19.766403: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773736099.964431      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773736100.020699      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773736100.452191      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773736100.452239      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773736100.452241      55 computation_placer.cc:177] computation placer alr


🚀 Autoencoder építése (1 GPU)...


I0000 00:00:1773736125.583623      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0



💾 Modell betöltése az inputból: /kaggle/input/models/bresgbor/spotify-autoencoder/keras/default/1/spotify_autoencoder.keras
✅ Súlyok betöltve!


/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:802: UserWarning: Skipping variable loading for optimizer 'loss_scale_optimizer', because it has 4 variables whereas the saved optimizer has 218 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))
/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:802: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 214 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


...

 Trainable params: 16,512,659 (62.99 MB)

 Non-trainable params: 4,240 (16.56 KB)


⚙️ Generátorok inicializálása...
✅ Generátor 'train': 21642 dal (Batch size: 64).
✅ Generátor 'val': 2705 dal (Batch size: 64).

🔥 P100 GPU TANÍTÁS INDÍTÁSA... (Batch size: 64)
Epoch 9/12


I0000 00:00:1773736167.005921     148 service.cc:152] XLA service 0x7e2031058dd0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1773736167.005963     148 service.cc:160]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1773736170.944594     148 cuda_dnn.cc:529] Loaded cuDNN version 91002
2026-03-17 08:30:03.073526: E external/local_xla/xla/service/slow_operation_alarm.cc:73] Trying algorithm eng2{k2=3,k3=0} for conv %cudnn-conv-bw-input.34 = (f32[64,16,384,1280]{3,2,1,0}, u8[0]{0}) custom-call(f32[64,1,384,1280]{3,2,1,0} %bitcast.40402, f32[1,16,3,3]{3,2,1,0} %bitcast.34862), window={size=3x3 pad=1_1x1_1}, dim_labels=bf01_oi01->bf01, custom_call_target="__cudnn$convBackwardInput", metadata={op_type="Conv2DBackpropInput" op_name="gradient_tape/spotify_autoencoder_1/tempo_output_1/convolution/Conv2DBackpropInput" source_file="/usr/local/lib/python3.12/dist-packages/tensorflow/python/framework/ops.py

339/339 ━━━━━━━━━━━━━━━━━━━━ 0s 8s/step - chroma_output_loss: 0.0604 - chroma_output_mae: 0.1954 - loss: 0.0353 - mel_output_loss: 0.0121 - mel_output_mae: 0.0850 - tempo_output_loss: 0.0034 - tempo_output_mae: 0.0343
Epoch 9: val_loss improved from inf to 0.04108, saving model to /kaggle/working/spotify_autoencoder.keras
339/339 ━━━━━━━━━━━━━━━━━━━━ 2775s 8s/step - chroma_output_loss: 0.0604 - chroma_output_mae: 0.1954 - loss: 0.0353 - mel_output_loss: 0.0121 - mel_output_mae: 0.0850 - tempo_output_loss: 0.0034 - tempo_output_mae: 0.0343 - val_chroma_output_loss: 0.0684 - val_chroma_output_mae: 0.2020 - val_loss: 0.0411 - val_mel_output_loss: 0.0146 - val_mel_output_mae: 0.0920 - val_tempo_output_loss: 0.0040 - val_tempo_output_mae: 0.0346
Epoch 10/12
339/339 ━━━━━━━━━━━━━━━━━━━━ 0s 7s/step - chroma_output_loss: 0.0604 - chroma_output_mae: 0.1951 - loss: 0.0345 - mel_output_loss: 0.0118 - mel_output_mae: 0.0841 - tempo_output_loss: 0.0031 - tempo_output_mae: 0.0304
Epoch 10: val_loss 

## Bridge

In [1]:
import os
import h5py
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, BatchNormalization, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping
from gensim.models import Word2Vec
from tqdm import tqdm

# --- ÚTVONALAK (Módosítsd a sajátodra!) ---
H5_PATH = "../Dataset/spotify_dataset_compressed.h5"
W2V_PATH = "../Models/song2vec.model" # Vagy a .bin fájlod, ha azt használod
AE_PATH = "../Models/spotify_autoencoder.keras"
SAVE_BRIDGE_PATH = "../Models/spotify_bridge_model.keras"

# --- CUSTOM RÉTEG ÉS LOSS ---
class L2NormLayer(tf.keras.layers.Layer):
    def call(self, x):
        return tf.math.l2_normalize(x, axis=1)

def cosine_loss(y_true, y_pred):
    return 1.0 - tf.reduce_mean(tf.reduce_sum(tf.math.l2_normalize(y_true, axis=1) * y_pred, axis=1))

def main():
    print("1. Modellek betöltése...")
    w2v_model = Word2Vec.load(W2V_PATH) # Ha a vektort használod: KeyedVectors.load_word2vec_format(...)
    
    # Teljes Autoencoder betöltése
    autoencoder = tf.keras.models.load_model(
        AE_PATH, 
        custom_objects={'cosine_loss': cosine_loss, 'L2NormLayer': L2NormLayer},
        compile=False # Nem akarjuk tovább tanítani, így gyorsabb
    )
    
    # ENCODER KINYERÉSE A TELJES MODELLBŐL!
    encoder = Model(inputs=autoencoder.inputs, outputs=autoencoder.get_layer('l2_norm').output)
    print("✅ Encoder sikeresen leválasztva!")

    print("\n2. Adatok kinyerése és párosítása (Ez eltarthat pár percig)...")
    X_latent = [] # Ide jönnek a 256 dimenziós AE vektorok
    y_target = [] # Ide jönnek a 400 dimenziós W2V vektorok
    
    with h5py.File(H5_PATH, 'r') as hf:
        uris = hf['ml/track_uri'][:]
        # Itt most a train és val halmazt használjuk a fordító betanításához!
        splits = hf['ml/split'][:]
        
        batch_size = 64
        total_samples = len(uris)
        
        for i in tqdm(range(0, total_samples, batch_size), desc="Vektorok kinyerése"):
            batch_uris = uris[i : i+batch_size]
            batch_splits = splits[i : i+batch_size]
            
            valid_indices = []
            for j, (u, s) in enumerate(zip(batch_uris, batch_splits)):
                uri = u.decode('utf-8') if isinstance(u, bytes) else u
                split = s.decode('utf-8') if isinstance(s, bytes) else s
                
                # Csak azokat használjuk, amik benne vannak a W2V-ben, és NEM a test halmazban vannak
                if uri in w2v_model.wv and split in ['train', 'val']:
                    valid_indices.append(j)
                    y_target.append(w2v_model.wv[uri])
            
            if not valid_indices:
                continue
                
            # Adatok betöltése a H5-ből
            actual_indices = [i + idx for idx in valid_indices]
            X_mel = np.expand_dims(hf['spectrograms/mel'][actual_indices], -1)
            X_chroma = np.expand_dims(hf['spectrograms/chroma'][actual_indices], -1)
            X_tempo = np.expand_dims(hf['spectrograms/tempogram'][actual_indices], -1)
            
            # Átküldjük az Encoderen
            latent_vectors = encoder.predict([X_mel, X_chroma, X_tempo], verbose=0)
            X_latent.extend(latent_vectors)
            
    X_latent = np.array(X_latent)
    y_target = np.array(y_target)
    
    print(f"\n✅ Kész! Összesen {len(X_latent)} dalt párosítottunk.")
    print(f"X (Bemenet) forma: {X_latent.shape} (Autoencoder látens vektor)")
    print(f"Y (Cél) forma: {y_target.shape} (Word2Vec vektor)")

    print("\n3. A Fordító (Bridge) modell építése...")
    # Egy egyszerű, de hatékony hálózat a 256 -> 400 konverzióhoz
    bridge_input = Input(shape=(256,), name='latent_input')
    x = Dense(512, activation='relu')(bridge_input)
    x = BatchNormalization()(x)
    x = Dropout(0.3)(x)
    x = Dense(512, activation='relu')(x)
    x = BatchNormalization()(x)
    x = Dropout(0.3)(x)
    
    raw_output = Dense(400, activation='linear')(x)
    bridge_output = L2NormLayer(name='l2_norm')(raw_output)
    
    bridge_model = Model(inputs=bridge_input, outputs=bridge_output, name="w2v_bridge")
    bridge_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3), loss=cosine_loss, metrics=['mae'])
    
    print("\n4. Fordító betanítása (VILLÁMGYORS LESZ)...")
    early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1)
    
    # 80-20% arányban felosztjuk az adatokat tanításra és validációra
    bridge_model.fit(
        X_latent, y_target,
        validation_split=0.2,
        batch_size=128,
        epochs=50,
        callbacks=[early_stop]
    )
    
    print("\n5. Fordító modell mentése...")
    bridge_model.save(SAVE_BRIDGE_PATH)
    print(f"✅ Mentve ide: {SAVE_BRIDGE_PATH}")

if __name__ == "__main__":
    main()

1. Modellek betöltése...

✅ Encoder sikeresen leválasztva!

2. Adatok kinyerése és párosítása (Ez eltarthat pár percig)...


Vektorok kinyerése: 100%|██████████| 423/423 [36:23<00:00,  5.16s/it]



✅ Kész! Összesen 24347 dalt párosítottunk.
X (Bemenet) forma: (24347, 256) (Autoencoder látens vektor)
Y (Cél) forma: (24347, 400) (Word2Vec vektor)

3. A Fordító (Bridge) modell építése...

4. Fordító betanítása (VILLÁMGYORS LESZ)...
Epoch 1/50
153/153 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - loss: 0.6263 - mae: 0.1585 - val_loss: 0.5296 - val_mae: 0.1542
Epoch 2/50
153/153 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.4878 - mae: 0.1526 - val_loss: 0.5165 - val_mae: 0.1537
Epoch 3/50
153/153 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.4739 - mae: 0.1520 - val_loss: 0.4879 - val_mae: 0.1524
Epoch 4/50
153/153 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.4610 - mae: 0.1515 - val_loss: 0.4585 - val_mae: 0.1511
Epoch 5/50
153/153 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.4520 - mae: 0.1511 - val_loss: 0.4478 - val_mae: 0.1506
Epoch 6/50
153/153 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.4481 - mae: 0.1509 - val_loss: 0.4468 - val_mae: 0.1505
Epoch 7/50
153/153 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - lo

In [3]:
import h5py
import numpy as np
import tensorflow as tf
from gensim.models import Word2Vec # Vagy KeyedVectors, ha a .bin-t használod
from collections import defaultdict
from tqdm import tqdm

# --- ÚTVONALAK ÉS BEÁLLÍTÁSOK ---
H5_PATH = "../Dataset/spotify_dataset.h5"
W2V_PATH = "../Models/song2vec.model" # vagy song2vec_vectors.bin
AE_PATH = "../Models/spotify_autoencoder.keras"
BRIDGE_PATH = "../Models/spotify_bridge_model.keras"
K_VALUES = [10, 20]
NUM_TEST_SAMPLES = 500

# Custom rétegek és loss
class L2NormLayer(tf.keras.layers.Layer):
    def call(self, x):
        return tf.math.l2_normalize(x, axis=1)

def cosine_loss(y_true, y_pred):
    return 1.0 - tf.reduce_mean(tf.reduce_sum(tf.math.l2_normalize(y_true, axis=1) * y_pred, axis=1))

def calculate_metrics(recommended_uris, relevant_uris, K):
    top_k = recommended_uris[:K]
    hits = len(set(top_k).intersection(relevant_uris))
    
    hit_rate = 1 if hits > 0 else 0
    precision = hits / K
    recall = hits / len(relevant_uris) if len(relevant_uris) > 0 else 0
    
    return hit_rate, precision, recall

def main():
    print("1. Modellek betöltése...")
    w2v_model = Word2Vec.load(W2V_PATH) 
    
    # --- AUTOENCODER BETÖLTÉSE ÉS VÁGÁSA ---
    autoencoder = tf.keras.models.load_model(
        AE_PATH, 
        custom_objects={'cosine_loss': cosine_loss, 'L2NormLayer': L2NormLayer},
        compile=False
    )
    # Leválasztjuk az Encodert a memóriában!
    from tensorflow.keras.models import Model
    encoder_model = Model(inputs=autoencoder.inputs, outputs=autoencoder.get_layer('l2_norm').output)
    
    # --- BRIDGE MODELL BETÖLTÉSE ---
    bridge_model = tf.keras.models.load_model(
        BRIDGE_PATH, 
        custom_objects={'cosine_loss': cosine_loss, 'L2NormLayer': L2NormLayer}
    )
    print("✅ Modellek és a híd sikeresen felépítve!")

    print("\n2. Ground Truth szótárak építése a RAM-ban (ez eltarthat 1-2 percig)...")
    track_to_playlists = defaultdict(set)
    playlist_to_tracks = defaultdict(set)
    
    with h5py.File(H5_PATH, "r") as hf:
        pids = hf["playlist_tracks/pid"][:]
        uris = hf["playlist_tracks/track_uri"][:]
        
        for i in tqdm(range(len(pids)), desc="Kapcsolatok feldolgozása"):
            pid = pids[i]
            uri = uris[i].decode('utf-8') if isinstance(uris[i], bytes) else uris[i]
            
            track_to_playlists[uri].add(pid)
            playlist_to_tracks[pid].add(uri)

    print("\n3. Teszt halmaz kinyerése...")
    with h5py.File(H5_PATH, "r") as hf:
        splits = hf["ml/split"][:]
        test_indices = [i for i, s in enumerate(splits) if (s.decode('utf-8') if isinstance(s, bytes) else s) == "test"]
        
        if NUM_TEST_SAMPLES and len(test_indices) > NUM_TEST_SAMPLES:
            np.random.seed(42)
            test_indices = np.random.choice(test_indices, NUM_TEST_SAMPLES, replace=False)
            
        print(f"Tesztelésre kiválasztva: {len(test_indices)} dal.")

        results = {"baseline": {k: {"hr": [], "prec": [], "rec": []} for k in K_VALUES},
                   "ae_bridge": {k: {"hr": [], "prec": [], "rec": []} for k in K_VALUES}}

        print("\n4. Kiértékelés futtatása...")
        for idx in tqdm(test_indices, desc="Dalok tesztelése"):
            uri = hf["ml/track_uri"][idx]
            uri = uri.decode('utf-8') if isinstance(uri, bytes) else uri
            
            if uri not in w2v_model.wv:
                continue
                
            # --- GROUND TRUTH KINYERÉSE ---
            pids = track_to_playlists.get(uri, set())
            if not pids: 
                continue 
                
            relevant_uris = set()
            for pid in pids:
                relevant_uris.update(playlist_to_tracks[pid])
            relevant_uris.discard(uri) 
            
            if not relevant_uris:
                continue

            # --- A) BASELINE: Sima Word2Vec predikció ---
            w2v_similars = w2v_model.wv.most_similar(uri, topn=max(K_VALUES))
            w2v_recs = [u for u, score in w2v_similars]

            # --- B) AUTOENCODER + BRIDGE PREDIKCIÓ ---
            mel = np.expand_dims(np.expand_dims(hf["spectrograms/mel"][idx], axis=0), axis=-1)
            chroma = np.expand_dims(np.expand_dims(hf["spectrograms/chroma"][idx], axis=0), axis=-1)
            tempo = np.expand_dims(np.expand_dims(hf["spectrograms/tempogram"][idx], axis=0), axis=-1)
            
            # 1. Lépés: Spektrogram -> 256 dimenzió
            latent_vector = encoder_model.predict([mel, chroma, tempo], verbose=0)
            
            # 2. Lépés: 256 dimenzió -> 400 dimenzió
            pred_vector = bridge_model.predict(latent_vector, verbose=0)[0]
            
            cnn_similars = w2v_model.wv.similar_by_vector(pred_vector, topn=max(K_VALUES))
            cnn_recs = [u for u, score in cnn_similars if u != uri][:max(K_VALUES)]
            
            if len(cnn_recs) < max(K_VALUES):
                extra = w2v_model.wv.similar_by_vector(pred_vector, topn=max(K_VALUES)+1)
                cnn_recs = [u for u, score in extra if u != uri][:max(K_VALUES)]

            # --- METRIKÁK KISZÁMÍTÁSA ---
            for k in K_VALUES:
                # Baseline
                hr_b, prec_b, rec_b = calculate_metrics(w2v_recs, relevant_uris, k)
                results["baseline"][k]["hr"].append(hr_b)
                results["baseline"][k]["prec"].append(prec_b)
                results["baseline"][k]["rec"].append(rec_b)
                
                # Autoencoder + Bridge
                hr_c, prec_c, rec_c = calculate_metrics(cnn_recs, relevant_uris, k)
                results["ae_bridge"][k]["hr"].append(hr_c)
                results["ae_bridge"][k]["prec"].append(prec_c)
                results["ae_bridge"][k]["rec"].append(rec_c)

    # --- 5. EREDMÉNYEK KIÍRÁSA ---
    print("\n" + "="*50)
    print("🏆 KIÉRTÉKELÉS EREDMÉNYE (Autoencoder + Bridge) 🏆")
    print("="*50)
    for k in K_VALUES:
        print(f"\n--- TOP {k} AJÁNLÁS ---")
        print("BASELINE (Tiszta Word2Vec - Felső korlát):")
        print(f"  Hit Rate@{k}:  {np.mean(results['baseline'][k]['hr'])*100:.2f}%")
        print(f"  Precision@{k}: {np.mean(results['baseline'][k]['prec'])*100:.2f}%")
        print(f"  Recall@{k}:    {np.mean(results['baseline'][k]['rec'])*100:.2f}%")
        
        print("\nAUTOENCODER + BRIDGE MODELL:")
        print(f"  Hit Rate@{k}:  {np.mean(results['ae_bridge'][k]['hr'])*100:.2f}%")
        print(f"  Precision@{k}: {np.mean(results['ae_bridge'][k]['prec'])*100:.2f}%")
        print(f"  Recall@{k}:    {np.mean(results['ae_bridge'][k]['rec'])*100:.2f}%")

if __name__ == "__main__":
    main()

1. Modellek betöltése...
✅ Modellek és a híd sikeresen felépítve!

2. Ground Truth szótárak építése a RAM-ban (ez eltarthat 1-2 percig)...


Kapcsolatok feldolgozása: 100%|██████████| 66346428/66346428 [02:01<00:00, 546109.50it/s]



3. Teszt halmaz kinyerése...
Tesztelésre kiválasztva: 500 dal.

4. Kiértékelés futtatása...


Dalok tesztelése: 100%|██████████| 500/500 [03:07<00:00,  2.66it/s]



🏆 KIÉRTÉKELÉS EREDMÉNYE (Autoencoder + Bridge) 🏆

--- TOP 10 AJÁNLÁS ---
BASELINE (Tiszta Word2Vec - Felső korlát):
  Hit Rate@10:  98.60%
  Precision@10: 85.00%
  Recall@10:    0.10%

AUTOENCODER + BRIDGE MODELL:
  Hit Rate@10:  40.60%
  Precision@10: 13.10%
  Recall@10:    0.01%

--- TOP 20 AJÁNLÁS ---
BASELINE (Tiszta Word2Vec - Felső korlát):
  Hit Rate@20:  99.40%
  Precision@20: 81.43%
  Recall@20:    0.19%

AUTOENCODER + BRIDGE MODELL:
  Hit Rate@20:  50.20%
  Precision@20: 12.73%
  Recall@20:    0.02%


## Kiértékelés

In [1]:
import os
import h5py
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model  
from gensim.models import Word2Vec
from collections import defaultdict
from tqdm import tqdm
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import roc_auc_score
import random

# --- ÚTVONALAK ÉS BEÁLLÍTÁSOK ---
H5_PATH = "../Dataset/spotify_dataset_compressed.h5"
W2V_PATH = "../Models/song2vec.model" 
AUTOENCODER_PATH = "../Models/spotify_autoencoder.keras" 

K_VALUES = [1, 5, 10, 20, 50, 100, 200, 500]
NUM_TEST_SAMPLES = 1000
STEP_SIZE = 2

# ==========================================
# 2. SEGÉDFÜGGVÉNYEK
# ==========================================
class L2NormLayer(tf.keras.layers.Layer):
    def call(self, x):
        return tf.math.l2_normalize(x, axis=1)

def calculate_metrics(recommended_uris, relevant_uris, K):
    top_k = recommended_uris[:K]
    hits = len(set(top_k).intersection(relevant_uris))
    
    hit_rate = 1 if hits > 0 else 0
    precision = hits / K
    recall = hits / len(relevant_uris) if len(relevant_uris) > 0 else 0
    
    return hit_rate, precision, recall

def calculate_auc_fast(sims_array, relevant_indices, valid_indices, num_negatives=1000):
    """
    Gyors AUC számítás, ami közvetlenül a kiszámolt hasonlósági mátrixot (sims_array) használja.
    """
    # 1. Pozitív pontszámok (a releváns dalok hasonlósága)
    pos_scores = [sims_array[idx] for idx in relevant_indices]
    
    if not pos_scores:
        return 0.5
        
    # 2. Negatív pontszámok (véletlenszerű, nem releváns dalok a Zárt Világból)
    neg_scores = []
    # Az érvényes indexekből kivonjuk a relevánsakat
    possible_negatives = list(set(valid_indices) - set(relevant_indices))
    
    n_neg = min(num_negatives, len(possible_negatives))
    
    if n_neg > 0:
        # Véletlenszerűen választunk negatívokat
        chosen_negatives = random.sample(possible_negatives, n_neg)
        neg_scores = [sims_array[idx] for idx in chosen_negatives]
    
    if not neg_scores:
        return 0.5
        
    # 3. ROC-AUC
    y_true = [1] * len(pos_scores) + [0] * len(neg_scores)
    y_scores = pos_scores + neg_scores
    
    try:
        return roc_auc_score(y_true, y_scores)
    except:
        return 0.5

# ==========================================
# 3. FŐPROGRAM
# ==========================================
def main():
    print("1. Modellek betöltése...")
    w2v_model = Word2Vec.load(W2V_PATH)
    
    full_autoencoder = tf.keras.models.load_model(
        AUTOENCODER_PATH, custom_objects={'L2NormLayer': L2NormLayer}, compile=False
    )

    encoder_model = Model(
        inputs=full_autoencoder.inputs, 
        outputs=full_autoencoder.get_layer('l2_norm').output
    )

    print("\n2. Zárt Világ (Closed World) halmaz meghatározása...")
    with h5py.File(H5_PATH, "r") as hf:
        all_uris_bytes = hf["ml/track_uri"][:]
        all_uris = np.array([u.decode('utf-8') if isinstance(u, bytes) else u for u in all_uris_bytes])
        splits = hf["ml/split"][:]
        splits_str = np.array([s.decode('utf-8') if isinstance(s, bytes) else s for s in splits])

    w2v_vocab = set(w2v_model.wv.key_to_index.keys())
    valid_uris = set(all_uris).intersection(w2v_vocab)
    
    # Készítünk egy gyors keresőt az érvényes URI-k indexeihez
    uri_to_index = {uri: i for i, uri in enumerate(all_uris)}
    valid_indices = [uri_to_index[uri] for uri in valid_uris]
    
    print(f" -> Érvényes dalok a Zárt Világban: {len(valid_uris):,} db")

    print("\n3. Akusztikus Adatbázisok (AE és Nyers) legenerálása...")
    ae_vectors = np.zeros((len(all_uris), 256), dtype=np.float32)
    raw_vectors_list = []
    
    with h5py.File(H5_PATH, "r") as hf:
        batch_size = 32 
        for i in tqdm(range(0, len(all_uris), batch_size), desc="Spektrogramok kódolása"):
            mel_batch = hf["spectrograms/mel"][i:i+batch_size]
            chroma_batch = hf["spectrograms/chroma"][i:i+batch_size]
            tempo_batch = hf["spectrograms/tempogram"][i:i+batch_size]
            
            mel_raw = np.mean(mel_batch, axis=1)
            chroma_raw = np.mean(chroma_batch, axis=1)
            tempo_raw = np.mean(tempo_batch, axis=1)
            
            raw_concat = np.concatenate([mel_raw, chroma_raw, tempo_raw], axis=1)
            raw_norms = np.linalg.norm(raw_concat, axis=1, keepdims=True)
            raw_concat_norm = np.divide(raw_concat, raw_norms, out=np.zeros_like(raw_concat), where=raw_norms!=0)
            raw_vectors_list.extend(raw_concat_norm)

            mel_ae = np.expand_dims(mel_batch, axis=-1).astype(np.float32)
            chroma_ae = np.expand_dims(chroma_batch, axis=-1).astype(np.float32)
            tempo_ae = np.expand_dims(tempo_batch, axis=-1).astype(np.float32)
            
            ae_vectors[i:i+batch_size] = encoder_model.predict([mel_ae, chroma_ae, tempo_ae], verbose=0)

    raw_vectors = np.array(raw_vectors_list, dtype=np.float32)

    # Word2Vec mátrix felépítése a gyors kereséshez (csak azokat, amik benne vannak a W2V-ben)
    w2v_matrix = np.zeros((len(all_uris), w2v_model.vector_size), dtype=np.float32)
    for i, uri in enumerate(all_uris):
        if uri in w2v_model.wv:
            w2v_matrix[i] = w2v_model.wv[uri]

    print("\n4. Ground Truth szótárak építése...")
    track_to_playlists = defaultdict(set)
    playlist_to_tracks = defaultdict(set)

    with h5py.File(H5_PATH, "r") as hf:
        pids = hf["playlist_tracks/pid"][:]
        gt_uris = hf["playlist_tracks/track_uri"][:]
        
        for i in tqdm(range(len(pids)), desc="Kapcsolatok feldolgozása"):
            pid = pids[i]
            u = gt_uris[i].decode('utf-8') if isinstance(gt_uris[i], bytes) else gt_uris[i]
            track_to_playlists[u].add(pid)
            playlist_to_tracks[pid].add(u)

    print("\n5. Teszt halmaz kinyerése...")
    test_indices = [i for i, s in enumerate(splits_str) if s == "test" and all_uris[i] in valid_uris]
    test_indices = test_indices[::STEP_SIZE]
    
    if NUM_TEST_SAMPLES and len(test_indices) > NUM_TEST_SAMPLES:
        test_indices = test_indices[:NUM_TEST_SAMPLES]

    results = {
        "baseline": {"auc": [], **{k: {"hr": [], "prec": [], "rec": []} for k in K_VALUES}},
        "raw":      {"auc": [], **{k: {"hr": [], "prec": [], "rec": []} for k in K_VALUES}},
        "ae":       {"auc": [], **{k: {"hr": [], "prec": [], "rec": []} for k in K_VALUES}}
    }

    print("\n6. Kiértékelés futtatása (Fair, Closed-World mód)...")
    for idx in tqdm(test_indices, desc="Dalok tesztelése"):
        query_uri = all_uris[idx]
        
        pids = track_to_playlists.get(query_uri, set())
        if not pids: continue
            
        relevant_uris = set()
        for pid in pids:
            relevant_uris.update(playlist_to_tracks[pid])
        relevant_uris.discard(query_uri)
        relevant_uris = relevant_uris.intersection(valid_uris)
        
        if not relevant_uris: continue
        
        # A releváns dalok indexei a gyorsított AUC-hez
        relevant_indices = [uri_to_index[u] for u in relevant_uris]

        # --- A) BASELINE (Word2Vec) ---
        query_w2v = w2v_matrix[idx].reshape(1, -1)
        sims_w2v = cosine_similarity(query_w2v, w2v_matrix)[0]
        sorted_indices_w2v = np.argsort(sims_w2v)[::-1]
        
        w2v_recs = []
        for i in sorted_indices_w2v:
            rec_uri = all_uris[i]
            if rec_uri in valid_uris and rec_uri != query_uri:
                w2v_recs.append(rec_uri)
            if len(w2v_recs) == max(K_VALUES): break
            
        results["baseline"]["auc"].append(calculate_auc_fast(sims_w2v, relevant_indices, valid_indices))

        # --- B) NYERS AUDIÓ BASELINE (RAW) ---
        query_raw = raw_vectors[idx].reshape(1, -1)
        sims_raw = cosine_similarity(query_raw, raw_vectors)[0]
        sorted_indices_raw = np.argsort(sims_raw)[::-1]
        
        raw_recs = []
        for i in sorted_indices_raw:
            rec_uri = all_uris[i]
            if rec_uri in valid_uris and rec_uri != query_uri:
                raw_recs.append(rec_uri)
            if len(raw_recs) == max(K_VALUES): break

        results["raw"]["auc"].append(calculate_auc_fast(sims_raw, relevant_indices, valid_indices))

        # --- C) 3-ÁGÚ AUTOENCODER KERESŐMOTOR ---
        query_ae = ae_vectors[idx].reshape(1, -1)
        sims_ae = cosine_similarity(query_ae, ae_vectors)[0]
        sorted_indices_ae = np.argsort(sims_ae)[::-1]
        
        ae_recs = []
        for i in sorted_indices_ae:
            rec_uri = all_uris[i]
            if rec_uri in valid_uris and rec_uri != query_uri:
                ae_recs.append(rec_uri)
            if len(ae_recs) == max(K_VALUES): break

        results["ae"]["auc"].append(calculate_auc_fast(sims_ae, relevant_indices, valid_indices))

        # --- METRIKÁK KISZÁMÍTÁSA ---
        for k in K_VALUES:
            hr_b, prec_b, rec_b = calculate_metrics(w2v_recs, relevant_uris, k)
            results["baseline"][k]["hr"].append(hr_b)
            results["baseline"][k]["prec"].append(prec_b)
            results["baseline"][k]["rec"].append(rec_b)
            
            hr_r, prec_r, rec_r = calculate_metrics(raw_recs, relevant_uris, k)
            results["raw"][k]["hr"].append(hr_r)
            results["raw"][k]["prec"].append(prec_r)
            results["raw"][k]["rec"].append(rec_r)

            hr_ae, prec_ae, rec_ae = calculate_metrics(ae_recs, relevant_uris, k)
            results["ae"][k]["hr"].append(hr_ae)
            results["ae"][k]["prec"].append(prec_ae)
            results["ae"][k]["rec"].append(rec_ae)

    # --- 7. EREDMÉNYEK KIÍRÁSA ---
    print("\n" + "="*75)
    print("🏆 KIÉRTÉKELÉS EREDMÉNYE (ZÁRT VILÁG - FAIR COMPARISON) 🏆")
    print("="*75)
    
    print(f"\n✨ GLOBÁLIS RANGSOROLÁSI MINŐSÉG (AUC):")
    print(f"  1. BASELINE (Word2Vec): {np.mean(results['baseline']['auc'])*100:05.2f}%")
    print(f"  2. NYERS AUDIÓ (Raw):   {np.mean(results['raw']['auc'])*100:05.2f}%")
    print(f"  3. AUTOENCODER (AE):    {np.mean(results['ae']['auc'])*100:05.2f}%")

    for k in K_VALUES:
        print(f"\n--- TOP {k} AJÁNLÁS ---")
        print("1. BASELINE (Word2Vec / Lejátszási Listák):")
        print(f"  Hit Rate@{k}:  {np.mean(results['baseline'][k]['hr'])*100:05.2f}% | Prec: {np.mean(results['baseline'][k]['prec'])*100:05.2f}% | Rec: {np.mean(results['baseline'][k]['rec'])*100:05.2f}%")
        
        print("2. NYERS AUDIÓ (Összefűzött Átlag Spektrogramok):")
        print(f"  Hit Rate@{k}:  {np.mean(results['raw'][k]['hr'])*100:05.2f}% | Prec: {np.mean(results['raw'][k]['prec'])*100:05.2f}% | Rec: {np.mean(results['raw'][k]['rec'])*100:05.2f}%")

        print("3. AUTOENCODER (Mélytanulásos Akusztika):")
        print(f"  Hit Rate@{k}:  {np.mean(results['ae'][k]['hr'])*100:05.2f}% | Prec: {np.mean(results['ae'][k]['prec'])*100:05.2f}% | Rec: {np.mean(results['ae'][k]['rec'])*100:05.2f}%")
    print("\n" + "="*75)

if __name__ == "__main__":
    main()

1. Modellek betöltése...


2. Zárt Világ (Closed World) halmaz meghatározása...
 -> Érvényes dalok a Zárt Világban: 25,511 db

3. Akusztikus Adatbázisok (AE és Nyers) legenerálása...


Spektrogramok kódolása: 100%|██████████| 846/846 [1:03:31<00:00,  4.50s/it]  



4. Ground Truth szótárak építése...


Kapcsolatok feldolgozása: 100%|██████████| 66346428/66346428 [02:04<00:00, 534233.72it/s]



5. Teszt halmaz kinyerése...

6. Kiértékelés futtatása (Fair, Closed-World mód)...


Dalok tesztelése: 100%|██████████| 1000/1000 [06:19<00:00,  2.63it/s]


🏆 KIÉRTÉKELÉS EREDMÉNYE (ZÁRT VILÁG - FAIR COMPARISON) 🏆

✨ GLOBÁLIS RANGSOROLÁSI MINŐSÉG (AUC):
  1. BASELINE (Word2Vec): 76.82%
  2. NYERS AUDIÓ (Raw):   50.40%
  3. AUTOENCODER (AE):    55.68%

--- TOP 1 AJÁNLÁS ---
1. BASELINE (Word2Vec / Lejátszási Listák):
  Hit Rate@1:  87.10% | Prec: 87.10% | Rec: 00.05%
2. NYERS AUDIÓ (Összefűzött Átlag Spektrogramok):
  Hit Rate@1:  22.20% | Prec: 22.20% | Rec: 00.01%
3. AUTOENCODER (Mélytanulásos Akusztika):
  Hit Rate@1:  35.20% | Prec: 35.20% | Rec: 00.02%

--- TOP 5 AJÁNLÁS ---
1. BASELINE (Word2Vec / Lejátszási Listák):
  Hit Rate@5:  96.30% | Prec: 79.28% | Rec: 00.22%
2. NYERS AUDIÓ (Összefűzött Átlag Spektrogramok):
  Hit Rate@5:  55.50% | Prec: 20.26% | Rec: 00.03%
3. AUTOENCODER (Mélytanulásos Akusztika):
  Hit Rate@5:  70.60% | Prec: 31.92% | Rec: 00.06%

--- TOP 10 AJÁNLÁS ---
1. BASELINE (Word2Vec / Lejátszási Listák):
  Hit Rate@10:  97.70% | Prec: 76.19% | Rec: 00.41%
2. NYERS AUDIÓ (Összefűzött Átlag Spektrogramok):
  Hit Rat

### Kimentett vektorttérrel

In [4]:
import os
import h5py
import numpy as np
from gensim.models import Word2Vec
from collections import defaultdict
from tqdm import tqdm
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import roc_auc_score
import random

# --- ÚTVONALAK ÉS BEÁLLÍTÁSOK ---
H5_PATH = "../Dataset/spotify_dataset_compressed.h5"
W2V_PATH = "../Models/song2vec.model" 
AE_VECTORS_PATH = "../Models/ae_vectors_closed_world.npy" # <-- ÚJ: A kimentett vektorok helye

K_VALUES = [1, 5, 10, 20, 50, 100, 200, 500]
NUM_TEST_SAMPLES = 1000
STEP_SIZE = 2

# ==========================================
# 1. SEGÉDFÜGGVÉNYEK
# ==========================================
def calculate_metrics(recommended_uris, relevant_uris, K):
    top_k = recommended_uris[:K]
    hits = len(set(top_k).intersection(relevant_uris))
    
    hit_rate = 1 if hits > 0 else 0
    precision = hits / K
    recall = hits / len(relevant_uris) if len(relevant_uris) > 0 else 0
    
    return hit_rate, precision, recall

def calculate_auc_fast(sims_array, relevant_indices, valid_indices, num_negatives=1000):
    pos_scores = [sims_array[idx] for idx in relevant_indices]
    if not pos_scores: return 0.5
        
    possible_negatives = list(set(valid_indices) - set(relevant_indices))
    n_neg = min(num_negatives, len(possible_negatives))
    
    neg_scores = []
    if n_neg > 0:
        chosen_negatives = random.sample(possible_negatives, n_neg)
        neg_scores = [sims_array[idx] for idx in chosen_negatives]
    
    if not neg_scores: return 0.5
        
    y_true = [1] * len(pos_scores) + [0] * len(neg_scores)
    y_scores = pos_scores + neg_scores
    
    try:
        return roc_auc_score(y_true, y_scores)
    except:
        return 0.5

# ==========================================
# 2. FŐPROGRAM
# ==========================================
def main():
    print("1. Word2Vec és AE vektorok betöltése...")
    w2v_model = Word2Vec.load(W2V_PATH)
    
    # --- ÚJ: Autoencoder helyett csak a mátrixot töltjük be ---
    ae_vectors = np.load(AE_VECTORS_PATH)
    print(f" -> Autoencoder vektorok betöltve. Alakja: {ae_vectors.shape}")

    print("\n2. Zárt Világ (Closed World) halmaz meghatározása...")
    with h5py.File(H5_PATH, "r") as hf:
        all_uris_bytes = hf["ml/track_uri"][:]
        all_uris = np.array([u.decode('utf-8') if isinstance(u, bytes) else u for u in all_uris_bytes])
        splits = hf["ml/split"][:]
        splits_str = np.array([s.decode('utf-8') if isinstance(s, bytes) else s for s in splits])

    w2v_vocab = set(w2v_model.wv.key_to_index.keys())
    valid_uris = set(all_uris).intersection(w2v_vocab)
    
    uri_to_index = {uri: i for i, uri in enumerate(all_uris)}
    valid_indices = [uri_to_index[uri] for uri in valid_uris]
    
    print(f" -> Érvényes dalok a Zárt Világban: {len(valid_uris):,} db")

    print("\n3. Nyers (Raw) Akusztikus adatbázis legenerálása...")
    # Az AE vektorok már megvannak, így itt csak a baseline "Raw" vektorokat számoljuk ki.
    raw_vectors_list = []
    
    with h5py.File(H5_PATH, "r") as hf:
        batch_size = 128 # Nagyobb batch size is lehet, mivel csak átlagot vonunk
        for i in tqdm(range(0, len(all_uris), batch_size), desc="Nyers jellemzők kinyerése"):
            mel_batch = hf["spectrograms/mel"][i:i+batch_size]
            chroma_batch = hf["spectrograms/chroma"][i:i+batch_size]
            tempo_batch = hf["spectrograms/tempogram"][i:i+batch_size]
            
            mel_raw = np.mean(mel_batch, axis=1)
            chroma_raw = np.mean(chroma_batch, axis=1)
            tempo_raw = np.mean(tempo_batch, axis=1)
            
            raw_concat = np.concatenate([mel_raw, chroma_raw, tempo_raw], axis=1)
            raw_norms = np.linalg.norm(raw_concat, axis=1, keepdims=True)
            raw_concat_norm = np.divide(raw_concat, raw_norms, out=np.zeros_like(raw_concat), where=raw_norms!=0)
            
            raw_vectors_list.extend(raw_concat_norm)

    raw_vectors = np.array(raw_vectors_list, dtype=np.float32)

    w2v_matrix = np.zeros((len(all_uris), w2v_model.vector_size), dtype=np.float32)
    for i, uri in enumerate(all_uris):
        if uri in w2v_model.wv:
            w2v_matrix[i] = w2v_model.wv[uri]

    print("\n4. Ground Truth szótárak építése...")
    track_to_playlists = defaultdict(set)
    playlist_to_tracks = defaultdict(set)

    with h5py.File(H5_PATH, "r") as hf:
        pids = hf["playlist_tracks/pid"][:]
        gt_uris = hf["playlist_tracks/track_uri"][:]
        
        for i in tqdm(range(len(pids)), desc="Kapcsolatok feldolgozása"):
            pid = pids[i]
            u = gt_uris[i].decode('utf-8') if isinstance(gt_uris[i], bytes) else gt_uris[i]
            track_to_playlists[u].add(pid)
            playlist_to_tracks[pid].add(u)

    print("\n5. Teszt halmaz kinyerése...")
    test_indices = [i for i, s in enumerate(splits_str) if s == "test" and all_uris[i] in valid_uris]
    test_indices = test_indices[::STEP_SIZE]
    
    if NUM_TEST_SAMPLES and len(test_indices) > NUM_TEST_SAMPLES:
        test_indices = test_indices[:NUM_TEST_SAMPLES]

    results = {
        "baseline": {"auc": [], **{k: {"hr": [], "prec": [], "rec": []} for k in K_VALUES}},
        "raw":      {"auc": [], **{k: {"hr": [], "prec": [], "rec": []} for k in K_VALUES}},
        "ae":       {"auc": [], **{k: {"hr": [], "prec": [], "rec": []} for k in K_VALUES}}
    }

    print("\n6. Kiértékelés futtatása (Fair, Closed-World mód)...")
    for idx in tqdm(test_indices, desc="Dalok tesztelése"):
        query_uri = all_uris[idx]
        
        pids = track_to_playlists.get(query_uri, set())
        if not pids: continue
            
        relevant_uris = set()
        for pid in pids:
            relevant_uris.update(playlist_to_tracks[pid])
        relevant_uris.discard(query_uri)
        relevant_uris = relevant_uris.intersection(valid_uris)
        
        if not relevant_uris: continue
        
        relevant_indices = [uri_to_index[u] for u in relevant_uris]

        # --- A) BASELINE (Word2Vec) ---
        query_w2v = w2v_matrix[idx].reshape(1, -1)
        sims_w2v = cosine_similarity(query_w2v, w2v_matrix)[0]
        sorted_indices_w2v = np.argsort(sims_w2v)[::-1]
        
        w2v_recs = [all_uris[i] for i in sorted_indices_w2v if all_uris[i] in valid_uris and all_uris[i] != query_uri][:max(K_VALUES)]
        results["baseline"]["auc"].append(calculate_auc_fast(sims_w2v, relevant_indices, valid_indices))

        # --- B) NYERS AUDIÓ BASELINE (RAW) ---
        query_raw = raw_vectors[idx].reshape(1, -1)
        sims_raw = cosine_similarity(query_raw, raw_vectors)[0]
        sorted_indices_raw = np.argsort(sims_raw)[::-1]
        
        raw_recs = [all_uris[i] for i in sorted_indices_raw if all_uris[i] in valid_uris and all_uris[i] != query_uri][:max(K_VALUES)]
        results["raw"]["auc"].append(calculate_auc_fast(sims_raw, relevant_indices, valid_indices))

        # --- C) 3-ÁGÚ AUTOENCODER KERESŐMOTOR ---
        # Itt már a betöltött numpy array-t használjuk közvetlenül!
        query_ae = ae_vectors[idx].reshape(1, -1)
        sims_ae = cosine_similarity(query_ae, ae_vectors)[0]
        sorted_indices_ae = np.argsort(sims_ae)[::-1]
        
        ae_recs = [all_uris[i] for i in sorted_indices_ae if all_uris[i] in valid_uris and all_uris[i] != query_uri][:max(K_VALUES)]
        results["ae"]["auc"].append(calculate_auc_fast(sims_ae, relevant_indices, valid_indices))

        # --- METRIKÁK KISZÁMÍTÁSA ---
        for k in K_VALUES:
            hr_b, prec_b, rec_b = calculate_metrics(w2v_recs, relevant_uris, k)
            results["baseline"][k]["hr"].append(hr_b)
            results["baseline"][k]["prec"].append(prec_b)
            results["baseline"][k]["rec"].append(rec_b)
            
            hr_r, prec_r, rec_r = calculate_metrics(raw_recs, relevant_uris, k)
            results["raw"][k]["hr"].append(hr_r)
            results["raw"][k]["prec"].append(prec_r)
            results["raw"][k]["rec"].append(rec_r)

            hr_ae, prec_ae, rec_ae = calculate_metrics(ae_recs, relevant_uris, k)
            results["ae"][k]["hr"].append(hr_ae)
            results["ae"][k]["prec"].append(prec_ae)
            results["ae"][k]["rec"].append(rec_ae)

    # --- 7. EREDMÉNYEK KIÍRÁSA ---
    print("\n" + "="*75)
    print("🏆 KIÉRTÉKELÉS EREDMÉNYE (ZÁRT VILÁG - FAIR COMPARISON) 🏆")
    print("="*75)
    
    print(f"\n✨ GLOBÁLIS RANGSOROLÁSI MINŐSÉG (AUC):")
    print(f"  1. BASELINE (Word2Vec): {np.mean(results['baseline']['auc'])*100:05.2f}%")
    print(f"  2. NYERS AUDIÓ (Raw):   {np.mean(results['raw']['auc'])*100:05.2f}%")
    print(f"  3. AUTOENCODER (AE):    {np.mean(results['ae']['auc'])*100:05.2f}%")

    for k in K_VALUES:
        print(f"\n--- TOP {k} AJÁNLÁS ---")
        print("1. BASELINE (Word2Vec / Lejátszási Listák):")
        print(f"  Hit Rate@{k}:  {np.mean(results['baseline'][k]['hr'])*100:05.2f}% | Prec: {np.mean(results['baseline'][k]['prec'])*100:05.2f}% | Rec: {np.mean(results['baseline'][k]['rec'])*100:05.2f}%")
        
        print("2. NYERS AUDIÓ (Összefűzött Átlag Spektrogramok):")
        print(f"  Hit Rate@{k}:  {np.mean(results['raw'][k]['hr'])*100:05.2f}% | Prec: {np.mean(results['raw'][k]['prec'])*100:05.2f}% | Rec: {np.mean(results['raw'][k]['rec'])*100:05.2f}%")

        print("3. AUTOENCODER (Mélytanulásos Akusztika):")
        print(f"  Hit Rate@{k}:  {np.mean(results['ae'][k]['hr'])*100:05.2f}% | Prec: {np.mean(results['ae'][k]['prec'])*100:05.2f}% | Rec: {np.mean(results['ae'][k]['rec'])*100:05.2f}%")
    print("\n" + "="*75)

if __name__ == "__main__":
    main()

1. Word2Vec és AE vektorok betöltése...
 -> Autoencoder vektorok betöltve. Alakja: (27052, 256)

2. Zárt Világ (Closed World) halmaz meghatározása...
 -> Érvényes dalok a Zárt Világban: 25,511 db

3. Nyers (Raw) Akusztikus adatbázis legenerálása...


Nyers jellemzők kinyerése: 100%|██████████| 212/212 [02:01<00:00,  1.74it/s]



4. Ground Truth szótárak építése...


Kapcsolatok feldolgozása: 100%|██████████| 66346428/66346428 [02:32<00:00, 435515.18it/s]



5. Teszt halmaz kinyerése...

6. Kiértékelés futtatása (Fair, Closed-World mód)...


Dalok tesztelése: 100%|██████████| 1000/1000 [09:46<00:00,  1.71it/s]


🏆 KIÉRTÉKELÉS EREDMÉNYE (ZÁRT VILÁG - FAIR COMPARISON) 🏆

✨ GLOBÁLIS RANGSOROLÁSI MINŐSÉG (AUC):
  1. BASELINE (Word2Vec): 76.84%
  2. NYERS AUDIÓ (Raw):   50.37%
  3. AUTOENCODER (AE):    55.70%

--- TOP 1 AJÁNLÁS ---
1. BASELINE (Word2Vec / Lejátszási Listák):
  Hit Rate@1:  87.10% | Prec: 87.10% | Rec: 00.05%
2. NYERS AUDIÓ (Összefűzött Átlag Spektrogramok):
  Hit Rate@1:  22.20% | Prec: 22.20% | Rec: 00.01%
3. AUTOENCODER (Mélytanulásos Akusztika):
  Hit Rate@1:  35.10% | Prec: 35.10% | Rec: 00.02%

--- TOP 5 AJÁNLÁS ---
1. BASELINE (Word2Vec / Lejátszási Listák):
  Hit Rate@5:  96.30% | Prec: 79.28% | Rec: 00.22%
2. NYERS AUDIÓ (Összefűzött Átlag Spektrogramok):
  Hit Rate@5:  55.50% | Prec: 20.26% | Rec: 00.03%
3. AUTOENCODER (Mélytanulásos Akusztika):
  Hit Rate@5:  70.70% | Prec: 32.04% | Rec: 00.06%

--- TOP 10 AJÁNLÁS ---
1. BASELINE (Word2Vec / Lejátszási Listák):
  Hit Rate@10:  97.70% | Prec: 76.19% | Rec: 00.41%
2. NYERS AUDIÓ (Összefűzött Átlag Spektrogramok):
  Hit Rat

In [3]:
import h5py
import numpy as np
from collections import defaultdict
from tqdm import tqdm
from gensim.models import Word2Vec

# --- BEÁLLÍTÁSOK ---
H5_PATH = "../Dataset/spotify_dataset_compressed.h5"
W2V_PATH = "../Models/song2vec.model" 
MIN_SEQUENCE_LENGTH = 5  # Hány dal kell minimum egy listába, hogy szekvenciának tekintsük?

def main():
    print("1. Zárt Világ (27k dal) betöltése...")
    w2v_model = Word2Vec.load(W2V_PATH)
    
    with h5py.File(H5_PATH, "r") as hf:
        all_uris_bytes = hf["ml/track_uri"][:]
        all_uris = np.array([u.decode('utf-8') if isinstance(u, bytes) else u for u in all_uris_bytes])

    w2v_vocab = set(w2v_model.wv.key_to_index.keys())
    valid_uris = set(all_uris).intersection(w2v_vocab)
    print(f" -> Érvényes dalok a Zárt Világban: {len(valid_uris):,} db\n")

    print("2. Lejátszási listák szűrése és sorrend megtartása...")
    # Itt sima listát (szekvenciát) használunk set helyett, hogy a sorrend megmaradjon!
    playlist_sequences = defaultdict(list)
    original_playlist_count = set()
    
    with h5py.File(H5_PATH, "r") as hf:
        pids = hf["playlist_tracks/pid"][:]
        uris = hf["playlist_tracks/track_uri"][:]
        
        for i in tqdm(range(len(pids)), desc="Szekvenciák építése"):
            pid = pids[i]
            uri = uris[i].decode('utf-8') if isinstance(uris[i], bytes) else uris[i]
            
            original_playlist_count.add(pid)
            
            # Csak akkor fűzzük hozzá a szekvenciához, ha benne van a Zárt Világban
            if uri in valid_uris:
                playlist_sequences[pid].append(uri)

    # 3. Statisztikák számítása
    print("\n3. Szekvenciális Statisztikák Elemzése...")
    total_original_playlists = len(original_playlist_count)
    
    # Szétválogatjuk a listákat a hosszuk alapján
    valid_sequences = {pid: seq for pid, seq in playlist_sequences.items() if len(seq) >= MIN_SEQUENCE_LENGTH}
    
    seq_lengths = [len(seq) for seq in valid_sequences.values()]
    
    print("\n" + "="*50)
    print("📊 CSONKÍTOTT SZEKVENCIA STATISZTIKÁK 📊")
    print("="*50)
    print(f"Eredeti lejátszási listák száma: {total_original_playlists:,}")
    print(f"Túlélő szekvenciák (min. {MIN_SEQUENCE_LENGTH} dallal): {len(valid_sequences):,} db")
    
    if len(seq_lengths) > 0:
        print(f"Túlélési arány: {(len(valid_sequences) / total_original_playlists) * 100:.2f}%")
        print(f"Átlagos szekvencia hossz: {np.mean(seq_lengths):.2f} dal")
        print(f"Maximális szekvencia hossz: {np.max(seq_lengths)} dal")
        print(f"Leggyakoribb (medián) hossz: {np.median(seq_lengths):.0f} dal")
        
        # Opcionális extra: Hány lista van, ami mondjuk legalább 10 hosszú?
        len_10_plus = sum(1 for l in seq_lengths if l >= 10)
        print(f"\nKomplex szekvenciák (min. 10 dallal): {len_10_plus:,} db")
    print("="*50)

if __name__ == "__main__":
    main()

1. Zárt Világ (27k dal) betöltése...
 -> Érvényes dalok a Zárt Világban: 25,511 db

2. Lejátszási listák szűrése és sorrend megtartása...


Szekvenciák építése: 100%|██████████| 66346428/66346428 [01:16<00:00, 865043.40it/s] 



3. Szekvenciális Statisztikák Elemzése...

📊 CSONKÍTOTT SZEKVENCIA STATISZTIKÁK 📊
Eredeti lejátszási listák száma: 1,000,000
Túlélő szekvenciák (min. 5 dallal): 883,155 db
Túlélési arány: 88.32%
Átlagos szekvencia hossz: 37.82 dal
Maximális szekvencia hossz: 223 dal
Leggyakoribb (medián) hossz: 27 dal

Komplex szekvenciák (min. 10 dallal): 762,012 db
